# 🩺 Tech Challenge Fase 3 - Fine-Tuning Biomédico com Unsloth (4-bit)

Este notebook executa o fine-tuning supervisionado (**SFT - Supervised Fine-Tuning**) de modelos de linguagem biomédicos utilizando a biblioteca **Unsloth** em quantização **4-bit** (QLoRA) no **Google Colab**.

### 📋 Objetivos e Fluxo de Execução
1. **Instalação e Setup Otimizado**: Instalar o Unsloth com suporte a FlashAttention / xFormers e baixo uso de VRAM.
2. **Conexão com Google Drive**: Montar o drive e carregar os datasets gerados na Fase de Processamento (`finetuning_train.jsonl` e `finetuning_val.jsonl`) na pasta `/Tech-Challenge_dataset/`.
3. **Catálogo de Modelos Unsloth 4-bit**: Seleção do **Llama 3.1 8B Instruct** (com lista de modelos alternativos disponíveis no catálogo do Unsloth).
4. **QLoRA com Unsloth**: Configuração de adaptadores de baixo posto (*Low-Rank Adaptation*) otimizados com `FastLanguageModel.get_peft_model`.
5. **Chat Template e Masking de Resposta**: Formatação para o formato Llama 3.1 e cálculo de perda (*loss*) focado exclusivamente nas respostas do assistente (`train_on_responses_only`).
6. **Treinamento Supervisionado com SFTTrainer**: Execução com monitoramento contínuo de perda de treino e validação.
7. **Validação Clínica e Qualitativa**: Avaliação inferencial em casos MedQuAD, PubMedQA e protocolos hospitalares, comparando com o *ground truth* e verificando citação de fontes e tom ético.
8. **Persistência no Google Drive**: Salvamento dos adaptadores LoRA diretamente no Google Drive para posterior inferência ou exportação para GGUF / Ollama.
9. **Publicação no Hugging Face Hub**: Autenticação com Hugging Face via token e upload do modelo fine-tuned (`llama-lora-medical-tc3`) para consumo direto via API do Hugging Face.

> ⚙️ **Configuração de Hardware Recomendada no Colab**:
> - Vá em **Ambiente de Execução (Runtime)** ➔ **Alterar tipo de ambiente de execução (Change runtime type)**.
> - Selecione **Acelerador de hardware: T4 GPU** (ou **A100 / L4 GPU** se disponível).

In [ ]:
# 1. Instalação do Unsloth e Dependências Otimizadas
# O Unsloth acelera o treinamento em até 2-5x e reduz o consumo de memória VRAM em até 70%.

!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps trl peft accelerate bitsandbytes
!pip install --quiet datasets matplotlib pandas seaborn huggingface_hub

print("✅ Unsloth e dependências instalados com sucesso!")

In [ ]:
# 2. Imports e Diagnóstico do Ambiente (GPU e VRAM)
import os
import json
import time
import random
from pathlib import Path
from typing import Dict, Any, List, Optional

import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from datasets import load_dataset
from huggingface_hub import login, notebook_login, HfApi, whoami, InferenceClient

import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer

# Verificação e diagnóstico da GPU
if not torch.cuda.is_available():
    raise SystemError(
        "❌ Nenhuma GPU detectada! Por favor, acerte o ambiente em: "
        "Runtime -> Change runtime type -> T4 GPU (ou A100/L4)."
    )

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"🚀 PyTorch Versão: {torch.__version__}")
print(f"⚡ GPU Ativa: {gpu_name} ({total_vram_gb:.2f} GB VRAM)")
print(f"🔥 Suporte a Bfloat16: {is_bfloat16_supported()}")
print(f"📦 Versão do Unsloth: {unsloth.__version__}")

In [ ]:
# 3. Montagem do Google Drive e Localização dos Datasets
# Conecta ao Google Drive onde os datasets 'finetuning_train.jsonl' e 'finetuning_val.jsonl' foram salvos.
from google.colab import drive

# 3.1 Monta o Google Drive
drive.mount('/content/drive')

# 3.2 Caminho base da pasta conforme especificado: /Tech-Challenge_dataset/
DRIVE_DIR = Path("/content/drive/MyDrive/Tech-Challenge_dataset")
TRAIN_PATH = DRIVE_DIR / "finetuning_train.jsonl"
VAL_PATH = DRIVE_DIR / "finetuning_val.jsonl"
OUTPUT_DIR = DRIVE_DIR / "lora_model"

print(f"📁 Pasta configurada: {DRIVE_DIR}")

# Verificação e busca automática de contingência (caso esteja em subpasta do Drive)
if not DRIVE_DIR.exists():
    print("🔍 Pasta padrão não encontrada em 'MyDrive/Tech-Challenge_dataset'. Buscando no Drive...")
    candidates = list(Path("/content/drive/MyDrive").glob("**/Tech-Challenge_dataset*"))
    if candidates:
        DRIVE_DIR = candidates[0]
        TRAIN_PATH = DRIVE_DIR / "finetuning_train.jsonl"
        VAL_PATH = DRIVE_DIR / "finetuning_val.jsonl"
        OUTPUT_DIR = DRIVE_DIR / "lora_model"
        print(f"✅ Pasta localizada com sucesso em: {DRIVE_DIR}")
    else:
        print("⚠️ Atenção: A pasta 'Tech-Challenge_dataset' ainda não foi encontrada no seu Meu Drive.")
        print("   Crie a pasta 'Tech-Challenge_dataset' e faça o upload de finetuning_train.jsonl e finetuning_val.jsonl.")

# Checagem de existência e integridade dos arquivos
if TRAIN_PATH.exists() and VAL_PATH.exists():
    train_mb = TRAIN_PATH.stat().st_size / (1024 * 1024)
    val_mb = VAL_PATH.stat().st_size / (1024 * 1024)
    train_count = sum(1 for _ in open(TRAIN_PATH, "r", encoding="utf-8"))
    val_count = sum(1 for _ in open(VAL_PATH, "r", encoding="utf-8"))
    print(f"✅ Arquivo de Treino: {TRAIN_PATH.name} ({train_mb:.2f} MB, {train_count:,} registros)")
    print(f"✅ Arquivo de Validação: {VAL_PATH.name} ({val_mb:.2f} MB, {val_count:,} registros)")
else:
    print(f"⚠️ Aviso: Verifique se os arquivos estão disponíveis no diretório:")
    print(f"   - {TRAIN_PATH}")
    print(f"   - {VAL_PATH}")

### 🤖 4. Catálogo de Modelos 4-bit Disponíveis no Unsloth

O Unsloth disponibiliza modelos pré-quantizados em 4 bits dinâmicos que preservam a acurácia de modelos 16-bit com redução drástica no uso de VRAM.

| ID | Nome do Modelo | VRAM Recomendada | Descrição |
|:---|:---|:---|:---|
| **1** | `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit` | **~6-8 GB (T4/A100)** | **Padrão Selecionado**. Estado da arte em 8B para instruções e raciocínio clínico. |
| 2 | `unsloth/Meta-Llama-3.1-8B-bnb-4bit` | ~6-8 GB | Versão base (sem alinhamento prévio de instruções). |
| 3 | `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` | ~3-4 GB | Modelo compacto excelente para fine-tuning rápido e dispositivos de ponta. |
| 4 | `unsloth/Llama-3.2-1B-Instruct-bnb-4bit` | ~2-3 GB | Ultraleve, ideal para prototipação e validações em lote. |
| 5 | `unsloth/mistral-7b-instruct-v0.3-bnb-4bit` | ~6-8 GB | Mistral 7B Instruct v0.3 com janela ampla de contexto. |
| 6 | `unsloth/Phi-3.5-mini-instruct` | ~4-5 GB | Modelo de 3.8B da Microsoft altamente competitivo em benchmarks. |
| 7 | `unsloth/gemma-2-9b-it-bnb-4bit` | ~8-10 GB | Gemma 2 9B do Google, excelente compreensão semântica. |
| 8 | `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` | ~6-8 GB | Qwen 2.5 7B com forte suporte multilíngue e código. |
| 9 | `unsloth/Qwen2.5-14B-Instruct-bnb-4bit` | ~12-14 GB (A100/L4) | Modelo maior para tarefas complexas de raciocínio. |

In [ ]:
# 4. Seleção e Configuração do Modelo Base
# Definimos o modelo Llama 3.1 8B Instruct 4-bit como padrão conforme o escopo do Tech Challenge.

AVAILABLE_MODELS = {
    "1": ("Llama 3.1 8B Instruct (Padrão Recomendado)", "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"),
    "2": ("Llama 3.1 8B Base", "unsloth/Meta-Llama-3.1-8B-bnb-4bit"),
    "3": ("Llama 3.2 3B Instruct (Ultra Leve)", "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"),
    "4": ("Llama 3.2 1B Instruct (Mais Rápido)", "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"),
    "5": ("Mistral 7B Instruct v0.3", "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"),
    "6": ("Phi-3.5 Mini Instruct (3.8B)", "unsloth/Phi-3.5-mini-instruct"),
    "7": ("Gemma 2 9B Instruct", "unsloth/gemma-2-9b-it-bnb-4bit"),
    "8": ("Qwen 2.5 7B Instruct", "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"),
    "9": ("Qwen 2.5 14B Instruct", "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"),
}

# Escolha do modelo (você pode alterar para qualquer ID da lista acima)
SELECTED_MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048  # Suporta até 8192 com RoPE interno automático do Unsloth
DTYPE = None           # None = detecção automática (Float16 na T4, Bfloat16 na A100/L4)
LOAD_IN_4BIT = True    # 4-bit QLoRA quantização ativada

# Token opcional do Hugging Face (se necessário para modelos protegidos por licença)
# os.environ["HF_TOKEN"] = "seu_token_hf_aqui"
HF_TOKEN = os.getenv("HF_TOKEN", None)

print(f"🎯 Modelo selecionado: {SELECTED_MODEL_ID}")
print(f"📏 Tamanho máximo de sequência: {MAX_SEQ_LENGTH}")
print(f"💾 Quantização 4-bit ativada: {LOAD_IN_4BIT}")

In [ ]:
# 5. Carregamento do Modelo e Tokenizer com Unsloth
print(f"⏳ Carregando {SELECTED_MODEL_ID} em 4 bits via FastLanguageModel...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = SELECTED_MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
    token = HF_TOKEN,
)

print("✅ Modelo base e Tokenizer carregados com sucesso!")

In [ ]:
# 6. Configuração dos Adaptadores LoRA / QLoRA
# Injeta os adaptadores Low-Rank Adaptation (LoRA) para treinar apenas matrizes de projeção do Attention e MLP.

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank LoRA recomendado (8, 16, 32, 64)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,      # Escala do LoRA
    lora_dropout = 0,     # 0 é otimizado no Unsloth
    bias = "none",         # "none" é otimizado no Unsloth
    use_gradient_checkpointing = "unsloth", # Economiza 30% de VRAM adicional
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

# Exibe contagem de parâmetros treináveis
trainable_params, all_params = model.get_nb_trainable_parameters()
percentage = (trainable_params / all_params) * 100
print(f"📊 Parâmetros Treináveis: {trainable_params:,} / {all_params:,} ({percentage:.2f}%)")

In [ ]:
# 7. Definição das Funções Auxiliares (Chat Template, Formatação e Inferência)

# 7.1 Configura o Chat Template padrão Llama 3.1 no tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

def formatting_prompts_func(batch: Dict[str, Any]) -> Dict[str, List[str]]:
    """
    Aplica o chat template oficial da Meta para o Llama 3.1 em cada amostra de conversação.
    Cada amostra possui a chave 'messages' com a lista [system, user, assistant].
    """
    conversations = batch["messages"]
    formatted_texts = [
        tokenizer.apply_chat_template(
            conversation,
            tokenize = False,
            add_generation_prompt = False
        )
        for conversation in conversations
    ]
    return {"text": formatted_texts}

def generate_medical_response(
    model_instance,
    user_question: str,
    system_prompt: Optional[str] = None,
    max_new_tokens: int = 512,
    temperature: float = 0.3,
) -> str:
    """
    Executa inferência com o modelo treinado, adicionando os tokens de controle e extraindo
    exclusivamente a resposta gerada pelo assistente.
    """
    default_system = (
        "You are a biomedical evidence assistant for healthcare professionals. "
        "Answer using the supplied scientific evidence and the hospital's internal protocols. "
        "Distinguish findings from interpretation, state limitations, and do not invent evidence. "
        "Always cite the source of the information used. "
        "Answer in the same language as the question. "
        "You are not intended to diagnose or prescribe: any treatment or prescription you mention "
        "is a suggestion that must be validated by the responsible physician."
    )
    
    messages = [
        {"role": "system", "content": system_prompt or default_system},
        {"role": "user", "content": user_question}
    ]
    
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt"
    ).to("cuda")
    
    outputs = model_instance.generate(
        input_ids = input_ids,
        max_new_tokens = max_new_tokens,
        temperature = temperature,
        top_p = 0.9,
        do_sample = True if temperature > 0 else False,
        pad_token_id = tokenizer.eos_token_id,
        use_cache = True,
    )
    
    # Remove os tokens de entrada e decodifica apenas os novos tokens gerados
    generated_tokens = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens = True).strip()

print("✅ Funções auxiliares e Chat Template configurados!")

In [ ]:
# 8. Carga dos Datasets e Aplicação do Chat Template
print(f"📂 Carregando dataset de treino: {TRAIN_PATH}")
raw_train = load_dataset("json", data_files=str(TRAIN_PATH), split="train")

print(f"📂 Carregando dataset de validação: {VAL_PATH}")
raw_val = load_dataset("json", data_files=str(VAL_PATH), split="train")

print(f"\n📊 Total de exemplos para Treino: {len(raw_train):,}")
print(f"📊 Total de exemplos para Validação: {len(raw_val):,}")

# Aplica o mapeamento do chat template em lote
print("🔄 Formatando conversações com o Chat Template Llama 3.1...")
train_dataset = raw_train.map(formatting_prompts_func, batched = True)
val_dataset = raw_val.map(formatting_prompts_func, batched = True)

# Inspeciona o formato final de uma amostra
print("\n" + "="*80)
print("🔍 Visualização de uma amostra formatada (Primeiros 600 caracteres):")
print("="*80)
print(train_dataset[0]["text"][:600] + "\n[...]")
print("="*80)

In [ ]:
# 9. Configuração do Treinamento Supervisionado (SFTTrainer e Masking)

# Hyperparâmetros configurados para estabilidade e alto rendimento em GPUs Colab
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    
    # Defina max_steps para um número controlado (ex: 60-120 para validação rápida),
    # ou comente max_steps e defina num_train_epochs = 1 para rodar todo o dataset.
    max_steps = 100,
    # num_train_epochs = 1,
    
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 5,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 42,
    output_dir = "outputs",
    
    # Configurações de validação contínua (loss train vs loss val)
    eval_strategy = "steps",
    eval_steps = 20,
    save_strategy = "steps",
    save_steps = 50,
    save_total_limit = 2,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False, # Packing acelera sequências curtas; False garante isolamento exato
    args = training_args,
)

# Treina EXCLUSIVAMENTE nas respostas do assistente (Response-Only Masking)
# Garante que a loss penalize apenas o output médico e não memorize os prompts do usuário
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

print("✅ SFTTrainer configurado com Response Masking do Llama 3.1!")

In [ ]:
# 10. Execução do Treinamento e Curvas de Perda (Loss)
gpu_stats = torch.cuda.get_device_properties(0)
start_vram_gb = round(torch.cuda.max_memory_reserved() / (1024**3), 2)
max_vram_gb = round(gpu_stats.total_memory / (1024**3), 2)

print(f"⚡ GPU: {gpu_stats.name} | VRAM Total: {max_vram_gb} GB")
print(f"⚡ VRAM reservada inicial: {start_vram_gb} GB")
print("🚀 Iniciando fine-tuning supervisionado...")

start_time = time.time()
trainer_stats = trainer.train()
elapsed_min = (time.time() - start_time) / 60

final_vram_gb = round(torch.cuda.max_memory_reserved() / (1024**3), 2)
print(f"\n🏁 Treinamento finalizado em: {elapsed_min:.2f} minutos!")
print(f"📊 VRAM máxima utilizada: {final_vram_gb} GB ({final_vram_gb/max_vram_gb*100:.1f}%)")

# Visualização da curva de perda (Treino vs Validação)
log_history = trainer.state.log_history
train_steps = [x["step"] for x in log_history if "loss" in x]
train_losses = [x["loss"] for x in log_history if "loss" in x]
eval_steps = [x["step"] for x in log_history if "eval_loss" in x]
eval_losses = [x["eval_loss"] for x in log_history if "eval_loss" in x]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label="Perda de Treino (Train Loss)", color="#1f77b4", linewidth=2)
if eval_losses:
    plt.plot(eval_steps, eval_losses, label="Perda de Validação (Eval Loss)", color="#ff7f0e", marker="o", linewidth=2)

plt.title("Curva de Aprendizado - Fine-Tuning Llama 3.1 8B (Biomédico)", fontsize=13)
plt.xlabel("Passos de Otimização (Steps)", fontsize=11)
plt.ylabel("Loss (Cross-Entropy)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# 11. Validação Qualitativa por Inferência Clínica
# Ativa a inferência ultra-rápida do Unsloth (2x mais rápida)
FastLanguageModel.for_inference(model)

# Casos de teste cobrindo as 3 fontes do Tech Challenge:
# 1. Protocolo institucional do hospital (português)
# 2. Pergunta clínica com evidência PubMedQA
# 3. Pergunta de saúde / patologia MedQuAD
clinical_test_cases = [
    {
        "origem": "Protocolo Hospitalar (Hospital Sintético)",
        "pergunta": "Qual é a conduta inicial recomendada no protocolo de dor torácica aguda em sala de emergência?"
    },
    {
        "origem": "PubMedQA (Evidência Científica)",
        "pergunta": (
            "Question: Does early administration of aspirin reduce vascular mortality in patients with acute myocardial infarction?\n\n"
            "Evidence (PubMed abstract): Multiple randomized controlled trials have shown that prompt oral administration "
            "of acetylsalicylic acid significantly decreases vascular mortality and non-fatal reinfarction in suspected acute myocardial infarction."
        )
    },
    {
        "origem": "MedQuAD (NIH - Conhecimento Médico Geral)",
        "pergunta": "What are the main causes and diagnostic criteria for adult Nephrotic Syndrome?"
    }
]

print("🔬 Executando inferências de validação clínica com o modelo ajustado:\n")
for i, case in enumerate(clinical_test_cases, start=1):
    print("=" * 85)
    print(f"📌 [Teste {i}] Fonte Alvo: {case['origem']}")
    print(f"❓ Pergunta do Usuário:\n{case['pergunta']}")
    
    response = generate_medical_response(model, case["pergunta"])
    
    print(f"\n💡 Resposta Gerada pelo Modelo Fine-Tuned:\n{response}")
    print("=" * 85 + "\n")

In [ ]:
# 12. Validação Comparativa e Métricas no Conjunto de Validação
# Seleciona amostras reais do 'finetuning_val.jsonl' para comparar a resposta gerada com o Ground Truth,
# verificando a presença de citações ('Source:') e a aderência ao tom ético/disclaimer.

num_eval_samples = 5
random.seed(42)
sample_indices = random.sample(range(len(raw_val)), num_eval_samples)

benchmark_results = []

print(f"🔎 Avaliando {num_eval_samples} casos reais de validação...\n")

for idx in sample_indices:
    messages = raw_val[idx]["messages"]
    user_query = next(m["content"] for m in messages if m["role"] == "user")
    expected_answer = next(m["content"] for m in messages if m["role"] == "assistant")
    
    # Gera a resposta do modelo
    generated_answer = generate_medical_response(model, user_query)
    
    # Checagens clínicas automáticas de conformidade
    has_citation = "source:" in generated_answer.lower() or "fonte:" in generated_answer.lower()
    has_disclaimer = any(
        term in generated_answer.lower()
        for term in ["médico", "physician", "profissional", "professional", "suggestion", "sugestão", "diagnose", "diagnóstico"]
    )
    
    benchmark_results.append({
        "ID": idx,
        "Pergunta": user_query[:100] + ("..." if len(user_query) > 100 else ""),
        "Resposta Esperada (Ground Truth)": expected_answer[:120] + "...",
        "Resposta Gerada (Modelo)": generated_answer[:120] + "...",
        "Cita Fonte?": "✅ Sim" if has_citation else "⚠️ Não",
        "Tom Ético/Disclaimer": "✅ Sim" if has_disclaimer else "⚠️ Não",
    })

df_results = pd.DataFrame(benchmark_results)
display(df_results)

In [ ]:
# 13. Salvamento dos Adaptadores LoRA no Google Drive e Exportação

# 13.1 Cria diretório no Google Drive e salva os pesos adaptados (LoRA weights)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"💾 Salvando pesos LoRA e Tokenizer no Google Drive em: {OUTPUT_DIR}")

model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("✅ Pesos LoRA salvos com sucesso no Google Drive!")

# 13.2 (Opcional) Salvar Modelo Mesclado Completo (Merged 16-bit)
# Descomente para exportar um modelo standalone sem necessidade de LoRA:
# MERGED_DIR = DRIVE_DIR / "merged_16bit_model"
# print(f"💾 Mesclando e salvando modelo completo em: {MERGED_DIR}")
# model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method = "merged_16bit")

# 13.3 (Opcional) Exportar em formato GGUF para executar localmente com Ollama ou Llama.cpp:
# GGUF_DIR = DRIVE_DIR / "gguf_model"
# print(f"💾 Exportando formato GGUF (q4_k_m) em: {GGUF_DIR}")
# model.save_pretrained_gguf(str(GGUF_DIR), tokenizer, quantization_method = "q4_k_m")

print("\n🎉 Pipeline de Fine-Tuning Biomédico finalizado com sucesso!")

### 🤗 14. Publicação e Consumo no Hugging Face Hub

Nesta seção, realizamos o upload do modelo adaptado (**LoRA**) diretamente para o **Hugging Face Hub** com o identificador `llama-lora-medical-tc3`.

Isso permite:
1. **Versionamento e Compartilhamento**: Manter o modelo salvo na nuvem do Hugging Face.
2. **Consumo Direto via API**: Usar a **Hugging Face Inference API** (Serverless ou Dedicated Endpoints).
3. **Integração Externa**: Carregar os pesos diretamente em aplicações externas (ex: backend FastAPI, LangChain, Streamlit) usando `PeftModel.from_pretrained`.

> 🔑 **Requisito**: Você precisará de um **Token de Acesso do Hugging Face** com permissão **Write**.
> Obtenha seu token gratuitamente em: [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
# 14. Autenticação no Hugging Face Hub
# Abre a conexão autenticada com o Hugging Face para permitir o upload do modelo.

from huggingface_hub import login, whoami

# Tenta obter token configurado em variável de ambiente ou Google Colab Secrets
hf_token = os.getenv("HF_TOKEN")
try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    print("🔑 Autenticando com HF_TOKEN pré-configurado...")
    login(token=hf_token)
else:
    print("👉 Solicitação de login no Hugging Face:")
    print("   (Cole seu token com permissão 'Write' obtido em https://huggingface.co/settings/tokens)")
    login()

# Valida e exibe a identidade autenticada
try:
    user_data = whoami()
    hf_username = user_data["name"]
    print(f"\n✅ Autenticado com sucesso no Hugging Face como: @{hf_username}")
except Exception as e:
    print(f"⚠️ Erro ao verificar credenciais: {e}")
    print("Por favor, execute novamente a célula e informe um token com permissão de escrita (Write).")

In [ ]:
# 15. Upload do Modelo Fine-Tuned para o Hugging Face Hub
# Sobe os adaptadores LoRA e os arquivos de configuração do Tokenizer para o repositório 'llama-lora-medical-tc3'.

HF_MODEL_NAME = "llama-lora-medical-tc3"
repo_id = f"{hf_username}/{HF_MODEL_NAME}"

print(f"🚀 Iniciando upload para o repositório: {repo_id}")
print("⏳ Enviando adaptadores LoRA e arquivos do Tokenizer...")

# 15.1 Upload dos pesos LoRA adaptados via Unsloth
model.push_to_hub(
    repo_id,
    private = False,  # Altere para True se desejar repositório privado
)

# 15.2 Upload explícito do Tokenizer (garante todos os metadados de chat template e vocabulário)
tokenizer.push_to_hub(
    repo_id,
    private = False,
)

print(f"\n🎉 Upload concluído com sucesso!")
print(f"🔗 Repositório disponível em: https://huggingface.co/{repo_id}")

In [ ]:
# 16. Exemplo de Uso e Consumo Direto via API do Hugging Face / Transformers
# Demonstração de como consumir o modelo 'llama-lora-medical-tc3' publicado no Hugging Face.

sample_question = "Qual a conduta inicial em dor torácica aguda segundo os protocolos hospitalares?"

print("=" * 85)
print(f"📋 GUIA DE CONSUMO EXTERNO DO MODELO: {repo_id}")
print("=" * 85)

# Exemplo de código Python para carregar o modelo em qualquer ambiente backend ou API:
python_code_sample = f"""
# --- Exemplo de consumo com PEFT e Transformers (FastAPI / Scripts externos) ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "{SELECTED_MODEL_ID}"
lora_repo_id = "{repo_id}"

# 1. Carrega o Tokenizer e o Modelo Base
tokenizer = AutoTokenizer.from_pretrained(lora_repo_id)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype = torch.float16,
    device_map = "auto",
)

# 2. Acopla os adaptadores LoRA treinados
model = PeftModel.from_pretrained(base_model, lora_repo_id)
model.eval()

# 3. Formata e gera a resposta
messages = [
    {{"role": "system", "content": "You are a biomedical evidence assistant for healthcare professionals."}},
    {{"role": "user", "content": "{sample_question}"}}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))
"""

print(python_code_sample)
print("=" * 85)
print("✨ Modelo pronto para ser consumido via Hugging Face Hub!")